In [1]:
import os
import sys

project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.append(project_root)

import pandas as pd
import mlflow
from src.storesales.pipeline import run_experiment

In [2]:
# MAIN PARAMETERS
ACTIVE_MODEL = "xgboost"

NUMERIC_SKEWED_FEATURES = []
NUMERIC_NORMAL_FEATURES = [
    "sales_lag15",
    "sales_lag16",
    "sales_lag17",
    "sales_lag18",
    "sales_lag19",
    "sales_lag20",
    "sales_lag364",
]
CATEGORICAL_FEATURES = []

TARGET = "sales"

COLUMNS_TO_KEEP = ["date", "family", "store_nbr", "is_int_sales"]

CUTOFFS_STR = ["2017-07-15", "2017-04-15", "2017-01-15", "2016-09-15"]
CUTOFFS_DATETIME = pd.to_datetime(CUTOFFS_STR)

HORIZON = 15

ISSUE_NAME = "sales_lag15-16-17-18-19-20-364_no_cat"
VERSION_NAME = "v0"

PATH_TRAIN_DF = "../data/processed/df_train.parquet"

In [3]:
df_results, trained_models = run_experiment(
    model_name=ACTIVE_MODEL,
    skewed_feats=NUMERIC_SKEWED_FEATURES,
    normal_feats=NUMERIC_NORMAL_FEATURES,
    cat_feats=CATEGORICAL_FEATURES,
    target=TARGET,
    cols_to_keep=COLUMNS_TO_KEEP,
    cutoffs=CUTOFFS_DATETIME,
    horizon=HORIZON,
    issue_name=ISSUE_NAME,
    version_name=VERSION_NAME,
    df_path=PATH_TRAIN_DF,
)

/Users/silviadeanituaaspiazu/Library/Caches/pypoetry/virtualenvs/storesales-timeseriesforecasting-5H9Z9Y-8-py3.15/lib/python3.15/site-packages/mlflow/data/dataset_source_registry.py:148: UserWarning: The specified dataset source can be interpreted in multiple ways: LocalArtifactDatasetSource, LocalArtifactDatasetSource. MLflow will assume that this is a LocalArtifactDatasetSource source.
  return _dataset_source_registry.resolve(
/Users/silviadeanituaaspiazu/Library/Caches/pypoetry/virtualenvs/storesales-timeseriesforecasting-5H9Z9Y-8-py3.15/lib/python3.15/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing va

Cutoff: 2017-07-15, MAE: 82.84, RMSE:337.64, RMSLE:0.46


2026/11/08 17:07:34 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/11/08 17:07:34 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Cutoff: 2017-04-15, MAE: 108.27, RMSE:451.12, RMSLE:0.73


2026/11/08 17:07:42 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/11/08 17:07:42 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Cutoff: 2017-01-15, MAE: 108.77, RMSE:501.85, RMSLE:0.49


2026/11/08 17:07:50 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/11/08 17:07:50 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Cutoff: 2016-09-15, MAE: 69.42, RMSE:262.46, RMSLE:0.50


2026/11/08 17:07:53 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/11/08 17:07:53 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


GLOBAL RMSLE: 0.5447


In [4]:
mlflow.end_run()

In [8]:
df = pd.read_parquet("../data/processed/df_train.parquet")
for cutoff, modelo in trained_models.items():
    # 1. Recuperamos el trozo de historia exacta que vio este modelo
    df_train_fold = df[df["date"] < cutoff]

    # 2. Filtramos las columnas correctas
    X_train = df_train_fold[modelo.feature_names_in_]

    # 3. La predicción máxima sobre su historia es su techo real de conocimiento
    techo_real = modelo.predict(X_train).max()

    print(f"{cutoff.date()}: {techo_real:.2f}")

2017-07-15: 12128.39
2017-04-15: 12061.76
2017-01-15: 13637.12
2016-09-15: 12305.78
